# Step 1 — Training-Free Baseline
DINOv2 / DINOv3 / SAM · argmax · multilayer · PCA-whitening · SPair-71k

In [ ]:
# Cell 0 — Mount Drive + create folder structure
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
DIRS = [
    f'{DRIVE_ROOT}/datasets/SPair-71k',
    f'{DRIVE_ROOT}/datasets/PF-Pascal',
    f'{DRIVE_ROOT}/datasets/PF-Willow',
    f'{DRIVE_ROOT}/datasets/AP-10K',
    f'{DRIVE_ROOT}/weights',
    f'{DRIVE_ROOT}/weights/finetuned',
    f'{DRIVE_ROOT}/weights/finetuned/lora',
    f'{DRIVE_ROOT}/results/step1',
    f'{DRIVE_ROOT}/results/step2',
    f'{DRIVE_ROOT}/results/step3',
    f'{DRIVE_ROOT}/results/step4/lora',
    f'{DRIVE_ROOT}/results/step4/mnn',
    f'{DRIVE_ROOT}/results/step4/ensemble',
    f'{DRIVE_ROOT}/results/step4/ap10k',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print('Folders ready.')

In [ ]:
# Cell 1 — Install packages + clone repo for model architectures
!pip install -q timm scikit-learn pandas segment-anything

import os, subprocess
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    # Replace URL with your actual repo URL
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/YOUR_USER/semantic_correspondence.git',
                    REPO_PATH], check=False)
    print('Repo cloned.')
else:
    print('Repo already present.')

import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('sys.path updated.')

In [ ]:
# Cell 2 — Download SPair-71k (guarded)
SPAIR_DIR = f'{DRIVE_ROOT}/datasets/SPair-71k'
SPAIR_TAR = f'{SPAIR_DIR}/SPair-71k.tar.gz'
SPAIR_CHECK = f'{SPAIR_DIR}/JPEGImages'

if not os.path.exists(SPAIR_CHECK):
    print('Downloading SPair-71k ...')
    !wget -q -O "{SPAIR_TAR}" http://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz
    !tar -xzf "{SPAIR_TAR}" -C "{SPAIR_DIR}"
    print('SPair-71k extracted.')
else:
    print('SPair-71k already present.')

In [ ]:
# Cell 3 — Download pretrained weights (guarded)
import subprocess

DINOV2_W = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
if not os.path.exists(DINOV2_W):
    print('Downloading DINOv2 weights...')
    subprocess.run(['wget', '-q', '-O', DINOV2_W,
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth'])
    print('DINOv2 weights saved.')
else:
    print('DINOv2 weights already present.')

SAM_W = f'{DRIVE_ROOT}/weights/sam_vit_b.pth'
if not os.path.exists(SAM_W):
    print('Downloading SAM ViT-B weights...')
    subprocess.run(['wget', '-q', '-O', SAM_W,
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'])
    print('SAM weights saved.')
else:
    print('SAM weights already present.')

DINOV3_W = f'{DRIVE_ROOT}/weights/dinov3_vitb16_pretrain.pth'
if not os.path.exists(DINOV3_W):
    print('DINOv3 weights NOT found.')
    print('Obtain dinov3_vitb16_pretrain.pth from the project maintainer')
    print(f'and save to: {DINOV3_W}')
else:
    print('DINOv3 weights already present.')

In [ ]:
# Cell 4 — Imports + config
import torch, numpy as np, json, time, os, math
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from collections import defaultdict
from sklearn.decomposition import PCA

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

SPAIR_BASE      = f'{DRIVE_ROOT}/datasets/SPair-71k/SPair-71k'
PAIR_ANN_PATH   = f'{SPAIR_BASE}/PairAnnotation'
LAYOUT_PATH     = f'{SPAIR_BASE}/Layout'
IMAGE_PATH      = f'{SPAIR_BASE}/JPEGImages'
DATASET_SIZE    = 'large'
PCK_ALPHA       = 0.1
THRESHOLDS      = [0.05, 0.1, 0.2]
RESULTS_DIR     = f'{DRIVE_ROOT}/results/step1'
IMG_SIZE_DINOV2 = 518
IMG_SIZE_DINOV3 = 512
IMG_SIZE_SAM    = 512

In [ ]:
# Cell 5 — Inline utility functions
# ----- Data utilities -----
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

import glob
class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.datatype = datatype
        self.pck_alpha = pck_alpha
        self.ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = self.ann_files[:len(self.ann_files) - 1]
        self.pair_ann_path = pair_ann_path
        self.image_path = image_path
        self.transform = Normalize(['src_img', 'trg_img'])
    def __len__(self): return len(self.ann_files)
    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            ann = json.load(f)
        category = ann['category']
        src_img = read_img(os.path.join(self.image_path, category, ann['src_imname']))
        trg_img = read_img(os.path.join(self.image_path, category, ann['trg_imname']))
        trg_bbox = ann['trg_bndbox']
        sample = {'src_imname': ann['src_imname'], 'trg_imname': ann['trg_imname'],
                  'src_imsize': src_img.size(), 'trg_imsize': trg_img.size(),
                  'trg_bbox': trg_bbox, 'category': category,
                  'src_img': src_img, 'trg_img': trg_img,
                  'src_kps': torch.tensor(ann['src_kps']).float(),
                  'trg_kps': torch.tensor(ann['trg_kps']).float(),
                  'kps_ids': ann['kps_ids']}
        return self.transform(sample)

# ----- Feature extraction -----
def extract_dense_features(model, img_tensor, training=False):
    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def extract_dense_features_multilayer(model, img_tensor, n_last_layers=3, training=False):
    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        layers = model.get_intermediate_layers(img_tensor, n=n_last_layers,
                                               return_class_token=False, norm=True)
        avg = torch.stack(layers, dim=0).mean(dim=0)
        B, N, D = avg.shape
        H = W = int(N ** 0.5)
        return avg.reshape(B, H, W, D)

def apply_pca_whitening(src_features, tgt_features, n_components=64):
    device_ = tgt_features.device
    _, H, W, D = tgt_features.shape
    n_pixels = H * W
    tgt_np = tgt_features.squeeze(0).reshape(n_pixels, D).cpu().float().numpy()
    src_np = src_features.squeeze(0).reshape(n_pixels, D).cpu().float().numpy()
    k = min(n_components, n_pixels, D)
    pca = PCA(n_components=k, whiten=True)
    pca.fit(tgt_np)
    tgt_pca = pca.transform(tgt_np)
    src_pca = pca.transform(src_np)
    tgt_out = torch.from_numpy(tgt_pca).float().reshape(1, H, W, k).to(device_)
    src_out = torch.from_numpy(src_pca).float().reshape(1, H, W, k).to(device_)
    return src_out, tgt_out

def extract_dense_features_SAM(model, img_tensor, training=False, image_size=512):
    import torch.nn as nn
    from torch.cuda.amp import autocast
    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        img_r = F.interpolate(img_tensor, size=(image_size, image_size), mode='bilinear', align_corners=False)
        if image_size != 1024:
            orig_pe = model.image_encoder.pos_embed
            b, oh, _, ed = orig_pe.shape
            ng = image_size // 16
            pe_r = F.interpolate(orig_pe.permute(0,3,1,2), size=(ng,ng),
                                 mode='bicubic', align_corners=False).permute(0,2,3,1)
            model.image_encoder.pos_embed = nn.Parameter(pe_r, requires_grad=training)
        if training:
            emb = model.image_encoder(img_r)
        else:
            with autocast():
                emb = model.image_encoder(img_r)
        if image_size != 1024:
            model.image_encoder.pos_embed = orig_pe
        return emb.permute(0, 2, 3, 1)

# ----- Coordinate utilities -----
def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

# ----- Matching -----
def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

# ----- PCK -----
def compute_pck_spair71k(pred_points, gt_points, bbox, threshold):
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    norm = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    nd   = dist / norm
    mask = nd <= threshold
    return float(np.mean(mask) * 100), mask, nd

# ----- Evaluation -----
def run_evaluate(model, dataset, device, thresholds, patch_size, resized_size,
                 use_multilayer=False, n_last_layers=3, use_pca=False, n_components=64,
                 early_stop=False):
    per_img, all_kp = [], []
    t0 = time.time()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                  size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                  size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features_multilayer(model, src_t, n_last_layers) if use_multilayer else extract_dense_features(model, src_t)
            tf = extract_dense_features_multilayer(model, tgt_t, n_last_layers) if use_multilayer else extract_dense_features(model, tgt_t)
            if use_pca:
                sf, tf = apply_pca_whitening(sf, tf, n_components)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sf_q = sf[0, py, px, :]
                sim = F.cosine_similarity(sf_q.unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, mask, nd = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks,
                            'src_imname': str(sample['src_imname']),
                            'trg_imname': str(sample['trg_imname'])})
            if (idx+1) % 200 == 0: print(f'  {idx+1} pairs...')
            if early_stop and idx >= 50: break
    return per_img, time.time() - t0

def save_exp_results(per_img, name, thresholds, results_dir):
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'  PCK@{thr:.2f}: {np.mean(pcks):.2f}% ± {np.std(pcks):.2f}%')
    out = {'name': name, 'n_pairs': len(per_img), 'stats': stats}
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f: json.dump(out, f, indent=2)
    print(f'  Saved → {path}')
    return stats

print('Utility functions loaded.')

In [ ]:
# Cell 6 — Load DINOv2 model
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base

dinov2 = dinov2_vit_base(
    img_size=(518, 518), patch_size=14,
    num_register_tokens=0, block_chunks=0, init_values=1.0
).to(device)
ckpt = torch.load(DINOV2_W, map_location=device)
dinov2.load_state_dict(ckpt, strict=True)
dinov2.eval()
print('DINOv2 loaded.')

test_ds = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='test')
print(f'Test set: {len(test_ds)} pairs')

In [ ]:
# Cell 7 — Exp 1.1: DINOv2 last layer + argmax
print('=== Exp 1.1: DINOv2 last layer + argmax ===')
res_11, t = run_evaluate(dinov2, test_ds, device, THRESHOLDS,
                          patch_size=14, resized_size=IMG_SIZE_DINOV2)
stats_11 = save_exp_results(res_11, 'exp1_1_dinov2_lastlayer_argmax', THRESHOLDS, RESULTS_DIR)
print(f'Time: {t:.1f}s')

In [ ]:
# Cell 8 — Exp 1.2: DINOv2 last 3 layers + argmax
print('=== Exp 1.2: DINOv2 last 3 layers + argmax ===')
res_12, t = run_evaluate(dinov2, test_ds, device, THRESHOLDS,
                          patch_size=14, resized_size=IMG_SIZE_DINOV2,
                          use_multilayer=True, n_last_layers=3)
stats_12 = save_exp_results(res_12, 'exp1_2_dinov2_multilayer_argmax', THRESHOLDS, RESULTS_DIR)
print(f'Time: {t:.1f}s')

In [ ]:
# Cell 9 — Exp 1.3: DINOv2 last 3 layers + PCA + argmax
print('=== Exp 1.3: DINOv2 last 3 layers + PCA(64) + argmax ===')
res_13, t = run_evaluate(dinov2, test_ds, device, THRESHOLDS,
                          patch_size=14, resized_size=IMG_SIZE_DINOV2,
                          use_multilayer=True, n_last_layers=3, use_pca=True, n_components=64)
stats_13 = save_exp_results(res_13, 'exp1_3_dinov2_multilayer_pca_argmax', THRESHOLDS, RESULTS_DIR)
print(f'Time: {t:.1f}s')

In [ ]:
# Cell 10 — Load DINOv3
try:
    from src.models.dinov3.dinov3.models.vision_transformer import vit_base as dinov3_vit_base
    dinov3 = dinov3_vit_base(
        img_size=(512, 512), patch_size=16,
        n_storage_tokens=4, layerscale_init=1.0, mask_k_bias=True
    ).to(device)
    ckpt3 = torch.load(DINOV3_W, map_location=device)
    dinov3.load_state_dict(ckpt3, strict=True)
    dinov3.eval()
    DINOV3_OK = True
    print('DINOv3 loaded.')
except Exception as e:
    print(f'DINOv3 not available ({e}). Skipping DINOv3 experiments.')
    DINOV3_OK = False

In [ ]:
# Cell 11 — Exp 1.4: DINOv3 last layer + argmax
if DINOV3_OK:
    print('=== Exp 1.4: DINOv3 last layer + argmax ===')
    res_14, t = run_evaluate(dinov3, test_ds, device, THRESHOLDS,
                              patch_size=16, resized_size=IMG_SIZE_DINOV3)
    stats_14 = save_exp_results(res_14, 'exp1_4_dinov3_lastlayer_argmax', THRESHOLDS, RESULTS_DIR)
    print(f'Time: {t:.1f}s')
else:
    stats_14 = None
    print('Skipped.')

In [ ]:
# Cell 12 — Exp 1.5: DINOv3 last 3 layers + argmax
if DINOV3_OK:
    print('=== Exp 1.5: DINOv3 multilayer + argmax ===')
    res_15, t = run_evaluate(dinov3, test_ds, device, THRESHOLDS,
                              patch_size=16, resized_size=IMG_SIZE_DINOV3,
                              use_multilayer=True, n_last_layers=3)
    stats_15 = save_exp_results(res_15, 'exp1_5_dinov3_multilayer_argmax', THRESHOLDS, RESULTS_DIR)
    print(f'Time: {t:.1f}s')
else:
    stats_15 = None
    print('Skipped.')

In [ ]:
# Cell 13 — Exp 1.6: DINOv3 multilayer + PCA + argmax
if DINOV3_OK:
    print('=== Exp 1.6: DINOv3 multilayer + PCA + argmax ===')
    res_16, t = run_evaluate(dinov3, test_ds, device, THRESHOLDS,
                              patch_size=16, resized_size=IMG_SIZE_DINOV3,
                              use_multilayer=True, n_last_layers=3, use_pca=True, n_components=64)
    stats_16 = save_exp_results(res_16, 'exp1_6_dinov3_multilayer_pca_argmax', THRESHOLDS, RESULTS_DIR)
    print(f'Time: {t:.1f}s')
else:
    stats_16 = None
    print('Skipped.')

In [ ]:
# Cell 14 — Load SAM
from src.models.segment_anything.segment_anything import sam_model_registry
sam = sam_model_registry['vit_b'](checkpoint=SAM_W).to(device)
sam.eval()
print('SAM loaded.')

In [ ]:
# Cell 15 — Exp 1.7: SAM + argmax
print('=== Exp 1.7: SAM (512) + argmax ===')

def run_evaluate_sam(model, dataset, device, thresholds, image_size=512, early_stop=False):
    per_img = []
    t0 = time.time()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = sample['src_img'].unsqueeze(0).to(device)
            tgt_t = sample['trg_img'].unsqueeze(0).to(device)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features_SAM(model, src_t, image_size=image_size)
            tf = extract_dense_features_SAM(model, tgt_t, image_size=image_size)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, 16, image_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, 16, image_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks,
                            'src_imname': str(sample['src_imname'])})
            if (idx+1) % 200 == 0: print(f'  {idx+1} pairs...')
            if early_stop and idx >= 50: break
    return per_img, time.time() - t0

res_17, t = run_evaluate_sam(sam, test_ds, device, THRESHOLDS)
stats_17 = save_exp_results(res_17, 'exp1_7_sam_argmax', THRESHOLDS, RESULTS_DIR)
print(f'Time: {t:.1f}s')

In [ ]:
# Cell 16 — Summary table
rows = [
    {'Exp': '1.1', 'Backbone': 'DINOv2', 'Method': 'Last layer + argmax', 'Stats': stats_11},
    {'Exp': '1.2', 'Backbone': 'DINOv2', 'Method': 'Multilayer + argmax',  'Stats': stats_12},
    {'Exp': '1.3', 'Backbone': 'DINOv2', 'Method': 'Multilayer+PCA+argmax','Stats': stats_13},
    {'Exp': '1.4', 'Backbone': 'DINOv3', 'Method': 'Last layer + argmax',  'Stats': stats_14},
    {'Exp': '1.5', 'Backbone': 'DINOv3', 'Method': 'Multilayer + argmax',  'Stats': stats_15},
    {'Exp': '1.6', 'Backbone': 'DINOv3', 'Method': 'Multilayer+PCA+argmax','Stats': stats_16},
    {'Exp': '1.7', 'Backbone': 'SAM',    'Method': 'Last layer + argmax',  'Stats': stats_17},
]
table = []
for r in rows:
    if r['Stats'] is None:
        table.append({'Exp': r['Exp'], 'Backbone': r['Backbone'], 'Method': r['Method'],
                      'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})
    else:
        table.append({'Exp': r['Exp'], 'Backbone': r['Backbone'], 'Method': r['Method'],
                      'PCK@0.05': f"{r['Stats'].get('pck@0.05',{}).get('mean',0):.2f}%",
                      'PCK@0.10': f"{r['Stats'].get('pck@0.10',{}).get('mean',0):.2f}%",
                      'PCK@0.20': f"{r['Stats'].get('pck@0.20',{}).get('mean',0):.2f}%"})
df = pd.DataFrame(table)
print('\n=== Step 1 Results ===')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/step1_summary.csv', index=False)
print(f'Summary saved to {RESULTS_DIR}/step1_summary.csv')